# Tomahawk Rev 2.3 — Axisymmetric Equilibrium and Ideal-MHD Feasibility Screen

This Kaggle notebook is a dependency-free numerical screen. It solves a fixed-boundary axisymmetric Grad–Shafranov equilibrium, checks the finite-difference residual and divergence of B, computes beta/current/q proxies, cross-checks D–T alpha heating with corrected bremsstrahlung units, and runs a reduced uncertainty sweep.

A passing result is an internal feasibility result for the stated equations and assumptions. It is not a validated 3-D MHD result, ignition proof, or hardware certification. Free-boundary coils, nonlinear ideal/resistive MHD, kinetic transport, turbulence, laser coupling, wall response, and experimental validation are outside this notebook.

In [ ]:
# CELL 1 — Kaggle-safe imports and settings
import json, math, os, platform, time, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
os.environ.setdefault("MPLCONFIGDIR", "/tmp/tomahawk-rev23-mpl")
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = int(os.environ.get("TOMAHAWK_MHD_SEED", "20260825"))
NR = int(os.environ.get("TOMAHAWK_MHD_NR", "161"))
NZ = int(os.environ.get("TOMAHAWK_MHD_NZ", "161"))
GS_ITERS = int(os.environ.get("TOMAHAWK_MHD_GS_ITERS", "12000"))
SMOKE_N = int(os.environ.get("TOMAHAWK_MHD_SMOKE_N", "2048"))
FULL_N = int(os.environ.get("TOMAHAWK_MHD_FULL_N", str(2**16)))
OUT = Path(os.environ.get("TOMAHAWK_MHD_OUTPUT_ROOT", "/kaggle/working/tomahawk_rev23_mhd"))
if not Path("/kaggle/working").exists():
    OUT = Path(os.environ.get("TOMAHAWK_MHD_OUTPUT_ROOT", str(Path.cwd() / "tomahawk_rev23_mhd")))
OUT = OUT.resolve()
OUT.mkdir(parents=True, exist_ok=True)
print("Python:", platform.python_version(), "grid:", f"{NR} x {NZ}", "full cases:", FULL_N)
print("Output:", OUT)

## 2. Reference inputs and explicit screening gates

B0 is the plasma on-axis field. Bpeak is the peak conductor field at the coil surface. The CAD plasma volume is kept separate from the simple circular-torus envelope.

The ideal-MHD thresholds are proxies selected for this preliminary test. Replace them with validated project requirements before using the result for design decisions.

In [ ]:
# CELL 2 — Inputs, constants, and R-Z grid
MU0 = 4.0e-7 * math.pi
KEV_TO_J = 1.0e3 * 1.602176634e-19
MEV_TO_J = 1.0e6 * 1.602176634e-19
AMU = 1.66053906660e-27
M_T = 3.01604928199 * AMU
E_FUSION = 17.6 * MEV_TO_J
ALPHA_FRAC, NEUTRON_FRAC = 3.5 / 17.6, 14.1 / 17.6

R0, A, KAPPA = 7.5, 3.0, 1.0
CAD_VOLUME = 950.0
B0, BPEAK = 14.0, 35.0
ND0, NT0 = 1.20e20, 1.20e20
TI0, TE0, TIE, TEE = 20.0, 20.0, 1.25, 1.25
GN, GT, ZEFF = 0.70, 0.80, 1.0
TAU_E = 1.36
ALPHA_DEPOSITION = 0.90
LASER_AVG = 25.0e6
LASER_COUPLING = 0.80
IP_REFERENCE_MA = 25.0

GS_RESIDUAL_TOL = 2.0e-3
DIVB_TOL = 2.0e-10
Q95_MIN, BETA_N_MAX, BETA_AXIS_MAX = 2.0, 3.5, 5.0
TAU_MARGIN_MIN = 0.20

R = np.linspace(R0 - 1.10 * A, R0 + 1.10 * A, NR)
Z = np.linspace(-1.10 * KAPPA * A, 1.10 * KAPPA * A, NZ)
RR, ZZ = np.meshgrid(R, Z, indexing="xy")
dR, dZ = float(R[1] - R[0]), float(Z[1] - Z[0])
rho2 = ((RR - R0) / A)**2 + (ZZ / (KAPPA * A))**2
rho = np.sqrt(np.maximum(rho2, 0.0))
MASK = rho2 <= 1.0
ME = np.zeros_like(MASK); ME[:, :-1] = MASK[:, 1:]
MW = np.zeros_like(MASK); MW[:, 1:] = MASK[:, :-1]
MN = np.zeros_like(MASK); MN[:-1, :] = MASK[1:, :]
MS = np.zeros_like(MASK); MS[1:, :] = MASK[:-1]
AE = 1.0 / dR**2 - 1.0 / (2.0 * RR * dR)
AW = 1.0 / dR**2 + 1.0 / (2.0 * RR * dR)
AN = np.full_like(RR, 1.0 / dZ**2)
AS = np.full_like(RR, 1.0 / dZ**2)
AP = -2.0 / dR**2 - 2.0 / dZ**2
P_AXIS_TARGET = (ND0 + NT0) * (TI0 + TE0) * KEV_TO_J

print(json.dumps({
    "R0_m": R0, "a_m": A, "kappa": KAPPA, "CAD_volume_m3": CAD_VOLUME,
    "B0_plasma_T": B0, "Bpeak_coil_T": BPEAK,
    "central_pressure_target_MPa": P_AXIS_TARGET / 1e6,
    "magnetic_pressure_axis_MPa": B0**2 / (2 * MU0) / 1e6,
    "magnetic_pressure_coil_MPa": BPEAK**2 / (2 * MU0) / 1e6,
    "grid_cells_inside": int(MASK.sum())
}, indent=2))

## 3. Fixed-boundary Grad–Shafranov solve

The equation solved is

Delta-star psi = -mu0 R squared times dp/dpsi - F dF/dpsi,

where Delta-star psi = d2psi/dR2 - (1/R) dpsi/dR + d2psi/dZ2. The boundary is held at psi equal to zero. Constant dp/dpsi and constant F dF/dpsi provide a transparent fixed-source equilibrium for this first screen.

In [ ]:
# CELL 3 — Finite-difference Grad–Shafranov solver and pressure calibration
def delta_star(psi):
    e = np.zeros_like(psi); w = np.zeros_like(psi)
    n = np.zeros_like(psi); s = np.zeros_like(psi)
    e[:, :-1] = np.where(ME[:, :-1], psi[:, 1:], 0.0)
    w[:, 1:] = np.where(MW[:, 1:], psi[:, :-1], 0.0)
    n[:-1, :] = np.where(MN[:-1, :], psi[1:, :], 0.0)
    s[1:, :] = np.where(MS[1:, :], psi[:-1, :], 0.0)
    return AE * e + AW * w + AN * n + AS * s + AP * psi

def solve_gs(dp_dpsi, ff_prime=0.0, omega=1.65, max_iter=GS_ITERS, tol=1.0e-6):
    source = np.where(MASK, -(MU0 * RR**2 * dp_dpsi + ff_prime), 0.0)
    scale = max(float(np.max(np.abs(source[MASK]))), 1.0)
    psi = np.zeros_like(RR)
    rel = float("inf")
    stop = max_iter
    checker = (np.indices(MASK.shape).sum(axis=0) & 1).astype(bool)
    for it in range(1, max_iter + 1):
        for red in (False, True):
            e = np.zeros_like(psi); w = np.zeros_like(psi)
            n = np.zeros_like(psi); s = np.zeros_like(psi)
            e[:, :-1] = np.where(ME[:, :-1], psi[:, 1:], 0.0)
            w[:, 1:] = np.where(MW[:, 1:], psi[:, :-1], 0.0)
            n[:-1, :] = np.where(MN[:-1, :], psi[1:, :], 0.0)
            s[1:, :] = np.where(MS[1:, :], psi[:-1, :], 0.0)
            trial = (source - AE*e - AW*w - AN*n - AS*s) / AP
            active = MASK & (checker == red)
            psi[active] = (1.0 - omega) * psi[active] + omega * trial[active]
        psi[~MASK] = 0.0
        if it == 1 or it % 25 == 0:
            rel = float(np.max(np.abs((delta_star(psi) - source)[MASK])) / scale)
            if rel < tol:
                stop = it
                break
    residual = delta_star(psi) - source
    rel = float(np.max(np.abs(residual[MASK])) / scale)
    return psi, source, residual, {"iterations": int(stop), "residual_relative": rel}

def calibrate_pressure(target_pa, ff_prime=0.0):
    dp = 1.0e5
    history = []
    for _ in range(10):
        trial, _, _, _ = solve_gs(dp, ff_prime, max_iter=1200, tol=3.0e-6)
        p = dp * float(np.max(trial[MASK]))
        history.append({"dp_dpsi": dp, "psi_axis_Wb": float(np.max(trial[MASK])), "p_axis_Pa": p})
        if not np.isfinite(p) or p <= 0.0:
            raise RuntimeError("Pressure calibration became non-positive.")
        dp *= math.sqrt(target_pa / p)
    psi, source, residual, info = solve_gs(dp, ff_prime)
    return dp, psi, source, residual, info, pd.DataFrame(history)

DP_DPSI, PSI, GS_SOURCE, GS_RESIDUAL, GS_INFO, CAL_HISTORY = calibrate_pressure(P_AXIS_TARGET)
PSI_AXIS = float(np.max(PSI[MASK]))
P_AXIS = DP_DPSI * PSI_AXIS
P_EQ = np.where(MASK, np.maximum(DP_DPSI * PSI, 0.0), 0.0)
print("GS iterations:", GS_INFO["iterations"])
print("GS relative residual:", GS_INFO["residual_relative"])
print("Solved p_axis (MPa):", P_AXIS / 1e6)

## 4. Reconstruct fields and calculate equilibrium/MHD proxies

In [ ]:
# CELL 4 — B field, current, q, beta, and divergence diagnostics
F_EDGE = R0 * B0
F = np.sqrt(np.maximum(F_EDGE**2, 0.0))
dpsi_dz, dpsi_dr = np.gradient(PSI, dZ, dR, edge_order=2)
BR = np.where(MASK, -dpsi_dz / RR, 0.0)
BZ = np.where(MASK, dpsi_dr / RR, 0.0)
BPHI = np.where(MASK, F / RR, 0.0)
JPHI = np.where(MASK, RR * DP_DPSI, 0.0)

dA = dR * dZ
V_GRID = float(np.sum((2.0 * math.pi * RR)[MASK]) * dA)
IP = float(np.sum(JPHI[MASK]) * dA)
IP95 = float(np.sum(JPHI[MASK & (rho <= 0.95)]) * dA)
Q95 = 2.0 * math.pi * (0.95 * A)**2 * B0 / (MU0 * R0 * max(IP95, 1.0))
P_VOL = float(np.sum(P_EQ[MASK] * (2.0 * math.pi * RR[MASK])) * dA / max(V_GRID, 1.0))
BETA_AXIS = 2.0 * MU0 * P_AXIS / B0**2
BETA_VOL = 2.0 * MU0 * P_VOL / B0**2
BETA_N = 100.0 * BETA_VOL * A * B0 / max(IP / 1e6, 1e-12)

DIVB = (np.gradient(RR * BR, dR, axis=1, edge_order=2) / RR
        + np.gradient(BZ, dZ, axis=0, edge_order=2))
CORE_MASK = MASK & (rho <= 0.90)
DIVB_SCALE = max(float(np.max(np.abs(BR[CORE_MASK]))), float(np.max(np.abs(BZ[CORE_MASK]))), 1.0) / A
DIVB_REL = float(np.max(np.abs(DIVB[CORE_MASK])) / DIVB_SCALE)

shells = np.linspace(0.05, 0.95, 19)
shell_p = np.array([np.nanmean(P_EQ[(rho >= x - 0.025) & (rho < x + 0.025)]) for x in shells])
PRESSURE_MONOTONIC = bool(np.all(np.diff(shell_p[np.isfinite(shell_p)]) <= 1e-6 * max(P_AXIS, 1.0)))

EQ_METRICS = {
    "gs_iterations": GS_INFO["iterations"],
    "gs_residual_relative": GS_INFO["residual_relative"],
    "divB_relative": DIVB_REL,
    "psi_axis_Wb": PSI_AXIS,
    "p_axis_MPa": P_AXIS / 1e6,
    "p_volume_average_MPa": P_VOL / 1e6,
    "I_p_solved_MA": IP / 1e6,
    "I_p_95_MA": IP95 / 1e6,
    "q95_cylindrical_proxy": Q95,
    "beta_axis_percent": 100.0 * BETA_AXIS,
    "beta_volume_percent": 100.0 * BETA_VOL,
    "beta_N_proxy": BETA_N,
    "pressure_monotonic_pass": PRESSURE_MONOTONIC,
    "grid_toroidal_volume_m3": V_GRID,
    "CAD_volume_m3": CAD_VOLUME,
}
print(json.dumps(EQ_METRICS, indent=2))

## 5. D–T power and confinement cross-check

The bremsstrahlung calculation explicitly converts electron temperature from keV to eV. This avoids the factor-of-sqrt(1000) unit error that can make radiation losses look artificially small.

In [ ]:
# CELL 5 — Radial profiles, D-T reactivity, alpha heating, and tau_E balance
def dt_reactivity(T):
    T = np.clip(np.asarray(T, dtype=float), 0.2, 100.0)
    BG, MRC2 = 34.3827, 1.124656e6
    C1,C2,C3,C4 = 1.17302e-9,1.51361e-2,7.51886e-2,4.60643e-3
    C5,C6,C7 = 1.35000e-2,-1.06750e-4,1.36600e-5
    num = T * (C2 + T * (C4 + T * C6))
    den = 1.0 + T * (C3 + T * (C5 + T * C7))
    theta = T / (1.0 - num / den)
    xi = (BG**2 / (4.0 * theta))**(1.0 / 3.0)
    return 1.0e-6 * C1 * theta * np.sqrt(xi / (MRC2 * T**3)) * np.exp(-3.0 * xi)

x, w = np.polynomial.legendre.leggauss(96)
rq, rw = 0.5 * (x + 1.0), 0.5 * w * (x + 1.0)
ri = pi = bi = ni_avg = 0.0
for rrho, weight in zip(rq, rw):
    ns = max(0.0, 1.0 - rrho**2)
    nD = ND0 * ns**GN
    nT = NT0 * ns**GN
    ni = nD + nT
    ne = ni
    Ti = TIE + (TI0 - TIE) * ns**GT
    Te = TEE + (TE0 - TEE) * ns**GT
    ri += weight * nD * nT * dt_reactivity(Ti)
    pi += weight * (ni * Ti + ne * Te) * KEV_TO_J
    bi += weight * ZEFF * ne * ni * np.sqrt(1000.0 * Te)
    ni_avg += weight * ni

reaction_rate = CAD_VOLUME * ri
P_FUSION = reaction_rate * E_FUSION
P_ALPHA = P_FUSION * ALPHA_FRAC
P_NEUTRON = P_FUSION * NEUTRON_FRAC
P_ALPHA_DEP = P_ALPHA * ALPHA_DEPOSITION
P_BREM = CAD_VOLUME * 5.35e-37 * bi
W_THERMAL = 1.5 * CAD_VOLUME * pi
P_TRANSPORT = W_THERMAL / TAU_E
P_AUX = LASER_AVG * LASER_COUPLING
ALPHA_RATIO = P_ALPHA_DEP / max(P_BREM + P_TRANSPORT, 1.0)
TAU_REQUIRED = W_THERMAL / max(P_ALPHA_DEP - P_BREM, 1.0) if P_ALPHA_DEP > P_BREM else float("inf")
TAU_MARGIN = TAU_E / TAU_REQUIRED - 1.0 if np.isfinite(TAU_REQUIRED) else -np.inf
Q_PLASMA = P_FUSION / max(P_AUX, 1.0)
TRIPLE_PRODUCT = ni_avg * 0.5 * (TI0 + TE0) * TAU_E

CONF_METRICS = {
    "fusion_GW_module": P_FUSION / 1e9,
    "alpha_GW_module": P_ALPHA / 1e9,
    "alpha_deposited_GW_module": P_ALPHA_DEP / 1e9,
    "neutron_GW_module": P_NEUTRON / 1e9,
    "bremsstrahlung_GW_module": P_BREM / 1e9,
    "transport_loss_GW_module": P_TRANSPORT / 1e9,
    "thermal_energy_GJ_module": W_THERMAL / 1e9,
    "absorbed_auxiliary_MW_module": P_AUX / 1e6,
    "Q_plasma": Q_PLASMA,
    "alpha_self_heating_ratio": ALPHA_RATIO,
    "tau_E_required_s": TAU_REQUIRED,
    "tau_E_margin_ratio": TAU_MARGIN,
    "triple_product_kev_s_m3": TRIPLE_PRODUCT,
}
print(json.dumps(CONF_METRICS, indent=2))

## 6. Gate results

The equilibrium numerical gates, ideal-MHD proxy gates, and alpha-heating gates are reported separately. This prevents a converged finite-difference field from being mislabeled as proof of ignition.

In [ ]:
# CELL 6 — Auditable gate table and reduced Monte Carlo screen
GATES = {
    "GS_residual_pass": GS_INFO["residual_relative"] <= GS_RESIDUAL_TOL,
    "divB_pass": DIVB_REL <= DIVB_TOL,
    "positive_pressure_pass": bool(P_AXIS > 0.0 and np.all(P_EQ[MASK] >= 0.0)),
    "pressure_monotonic_pass": PRESSURE_MONOTONIC,
    "q95_proxy_pass": bool(Q95 >= Q95_MIN),
    "beta_N_proxy_pass": bool(BETA_N <= BETA_N_MAX),
    "beta_axis_proxy_pass": bool(100.0 * BETA_AXIS <= BETA_AXIS_MAX),
    "alpha_self_heating_pass": bool(ALPHA_RATIO >= 1.0),
    "tau_E_margin_pass": bool(TAU_MARGIN >= TAU_MARGIN_MIN),
}
GATES["equilibrium_numerical_pass"] = all(GATES[x] for x in ("GS_residual_pass", "divB_pass", "positive_pressure_pass"))
GATES["ideal_MHD_proxy_pass"] = all(GATES[x] for x in ("pressure_monotonic_pass", "q95_proxy_pass", "beta_N_proxy_pass", "beta_axis_proxy_pass"))
GATES["confinement_feasibility_screen_pass"] = all(GATES[x] for x in ("equilibrium_numerical_pass", "ideal_MHD_proxy_pass", "alpha_self_heating_pass", "tau_E_margin_pass"))
print(json.dumps(GATES, indent=2))

RANGES = {
    "p_axis_scale": (0.70, 1.40), "B0_T": (12.0, 16.0), "a_m": (2.6, 3.4),
    "R0_m": (6.8, 8.2), "Ip_MA": (18.0, 32.0), "tau_E_s": (0.60, 3.00),
    "density_scale": (0.80, 1.30), "temperature_scale": (0.80, 1.30),
    "Z_eff": (1.00, 1.80), "alpha_deposition": (0.80, 0.98),
}

def reduced_screen(n, seed, label):
    rng = np.random.default_rng(seed)
    u = rng.random((n, len(RANGES)))
    s = {}
    for j, (name, (lo, hi)) in enumerate(RANGES.items()):
        s[name] = lo + u[:, j] * (hi - lo)
    p = P_AXIS_TARGET * s["p_axis_scale"]
    fusion = P_FUSION * s["density_scale"]**2 * s["temperature_scale"]**1.5
    alpha_dep = fusion * ALPHA_FRAC * s["alpha_deposition"]
    brem = P_BREM * s["density_scale"]**2 * np.sqrt(s["temperature_scale"]) * s["Z_eff"]
    thermal = W_THERMAL * s["density_scale"] * s["temperature_scale"]
    transport = thermal / s["tau_E_s"]
    ratio = alpha_dep / np.maximum(brem + transport, 1.0)
    tau_req = np.where(alpha_dep > brem, thermal / np.maximum(alpha_dep - brem, 1.0), np.inf)
    margin = np.where(np.isfinite(tau_req), s["tau_E_s"] / tau_req - 1.0, -np.inf)
    beta_axis = 100.0 * 2.0 * MU0 * p / s["B0_T"]**2
    beta_n = 100.0 * BETA_VOL * s["p_axis_scale"] * s["a_m"] * s["B0_T"] / np.maximum(s["Ip_MA"], 1e-9)
    q95 = 2.0 * math.pi * (0.95 * s["a_m"])**2 * s["B0_T"] / (MU0 * s["R0_m"] * s["Ip_MA"] * 1e6)
    frame = pd.DataFrame({**s, "fusion_GW_module": fusion / 1e9, "alpha_self_heating_ratio": ratio,
        "tau_E_required_s": tau_req, "tau_E_margin_ratio": margin, "beta_axis_percent": beta_axis,
        "beta_N_proxy": beta_n, "q95_proxy": q95,
        "GS_residual_baseline": GS_INFO["residual_relative"], "divB_baseline": DIVB_REL})
    frame["gate_all"] = ((frame.GS_residual_baseline <= GS_RESIDUAL_TOL) &
        (frame.divB_baseline <= DIVB_TOL) & (frame.q95_proxy >= Q95_MIN) &
        (frame.beta_N_proxy <= BETA_N_MAX) & (frame.beta_axis_percent <= BETA_AXIS_MAX) &
        (frame.alpha_self_heating_ratio >= 1.0) & (frame.tau_E_margin_ratio >= TAU_MARGIN_MIN))
    out = OUT / label; out.mkdir(parents=True, exist_ok=True)
    frame.to_csv(out / "reduced_screen_cases.csv", index=False)
    frame.nlargest(min(1000, len(frame)), "alpha_self_heating_ratio").to_csv(out / "top_alpha_cases.csv", index=False)
    tau_values = frame["tau_E_margin_ratio"].replace([np.inf, -np.inf], np.nan).dropna()
    best_tau = float(tau_values.max()) if len(tau_values) else None
    summary = {"label": label, "cases": int(n), "all_gate_passes": int(frame.gate_all.sum()),
        "all_gate_pass_percent": float(frame.gate_all.mean() * 100.0),
        "best_alpha_self_heating_ratio": float(frame.alpha_self_heating_ratio.max()),
        "best_tau_E_margin_ratio": best_tau,
        "best_q95_proxy": float(frame.q95_proxy.max()),
        "note": "Reduced scaling screen; not a separate GS solve for every sample."}
    (out / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
    print(label, json.dumps(summary, indent=2))
    return frame, summary

SMOKE_RESULT, SMOKE_SUMMARY = reduced_screen(SMOKE_N, SEED, "smoke")
FULL_RESULT, FULL_SUMMARY = reduced_screen(FULL_N, SEED + 1, "full")

## 7. Plots, manifest, and downloadable bundle

In [ ]:
# CELL 7 — Export all diagnostics and create one ZIP bundle
fig, ax = plt.subplots(1, 3, figsize=(16, 4.8), constrained_layout=True)
levels = np.linspace(0.0, max(float(np.nanmax(PSI[MASK])), 1.0), 18)
cs = ax[0].contour(RR, ZZ, np.where(MASK, PSI, np.nan), levels=levels, cmap="viridis")
ax[0].clabel(cs, inline=True, fontsize=7, fmt="%.1f"); ax[0].set_title("Poloidal flux")
im = ax[1].pcolormesh(RR, ZZ, np.where(MASK, P_EQ / 1e6, np.nan), shading="auto", cmap="magma")
fig.colorbar(im, ax=ax[1], label="MPa"); ax[1].set_title("Pressure")
im = ax[2].pcolormesh(RR, ZZ, np.where(MASK, JPHI / 1e6, np.nan), shading="auto", cmap="coolwarm")
fig.colorbar(im, ax[2], label="MA/m2"); ax[2].set_title("Toroidal current density")
for axy in ax:
    axy.set_xlabel("R (m)"); axy.set_ylabel("Z (m)")
fig.savefig(OUT / "equilibrium_fields.png", dpi=170); plt.close(fig)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.8), constrained_layout=True)
ax[0].scatter(FULL_RESULT.q95_proxy, FULL_RESULT.beta_N_proxy, s=3, alpha=0.2)
ax[0].axvline(Q95_MIN, color="black", ls="--"); ax[0].axhline(BETA_N_MAX, color="black", ls="--")
ax[0].set_xlabel("q95 proxy"); ax[0].set_ylabel("beta_N proxy"); ax[0].set_title("MHD proxy space")
vals = FULL_RESULT.alpha_self_heating_ratio.replace([np.inf, -np.inf], np.nan).dropna()
ax[1].hist(vals, bins=60, color="#6a3dad", alpha=0.85); ax[1].axvline(1.0, color="red", ls="--")
ax[1].set_xlabel("deposited alpha / modeled losses"); ax[1].set_title("Confinement margin")
fig.savefig(OUT / "mhd_confinement_sweep.png", dpi=170); plt.close(fig)

pd.DataFrame([EQ_METRICS]).to_csv(OUT / "equilibrium_metrics.csv", index=False)
pd.DataFrame([CONF_METRICS]).to_csv(OUT / "confinement_metrics.csv", index=False)
(OUT / "equilibrium_metrics.json").write_text(json.dumps(EQ_METRICS, indent=2), encoding="utf-8")
(OUT / "confinement_metrics.json").write_text(json.dumps(CONF_METRICS, indent=2), encoding="utf-8")
(OUT / "gate_results.json").write_text(json.dumps(GATES, indent=2), encoding="utf-8")
CAL_HISTORY.to_csv(OUT / "pressure_calibration_history.csv", index=False)
np.savez_compressed(OUT / "equilibrium_fields.npz", R=R, Z=Z, psi=PSI, pressure=P_EQ, BR=BR, BZ=BZ, BPHI=BPHI, JPHI=JPHI, rho=rho)

MANIFEST = {"model_revision": "2.3-axisymmetric-equilibrium-mhd-screen",
    "status": "preliminary numerical feasibility screen only", "python": platform.python_version(),
    "numpy": np.__version__, "pandas": pd.__version__, "equilibrium_metrics": EQ_METRICS,
    "confinement_metrics": CONF_METRICS, "gates": GATES, "smoke_summary": SMOKE_SUMMARY,
    "full_summary": FULL_SUMMARY, "limitations": [
        "Fixed-boundary axisymmetric Grad-Shafranov solve; no free-boundary coils.",
        "Constant dp/dpsi and F dF/dpsi source functions.",
        "q95 and beta_N are proxies, not a stability eigenvalue calculation.",
        "Reduced Monte Carlo samples do not re-solve Grad-Shafranov.",
        "No nonlinear MHD, kinetic transport, neutronics, materials, or experiment."
    ]}
(OUT / "manifest.json").write_text(json.dumps(MANIFEST, indent=2), encoding="utf-8")
zip_path = OUT / "Tomahawk_Rev23_Equilibrium_MHD_Screen.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for p in sorted(OUT.rglob("*")):
        if p.is_file() and p != zip_path:
            z.write(p, p.relative_to(OUT))

print("\nFINAL SCREEN STATUS")
print(json.dumps({"equilibrium_numerical_pass": GATES["equilibrium_numerical_pass"],
    "ideal_MHD_proxy_pass": GATES["ideal_MHD_proxy_pass"],
    "alpha_self_heating_pass": GATES["alpha_self_heating_pass"],
    "confinement_feasibility_screen_pass": GATES["confinement_feasibility_screen_pass"],
    "full_reduced_screen_pass_percent": FULL_SUMMARY["all_gate_pass_percent"],
    "bundle": str(zip_path)}, indent=2))

## Interpretation

A numerical equilibrium pass means the finite-difference Grad–Shafranov residual and div(B) check meet the declared tolerances. An ideal-MHD proxy pass means q95, normalized beta, pressure monotonicity, and axis beta meet the selected screening thresholds. The combined confinement screen additionally requires alpha self-heating to cover modeled bremsstrahlung plus transport losses with the declared tau_E margin.

None of these labels substitutes for a validated free-boundary equilibrium, nonlinear ideal/resistive MHD stability spectrum, transport validation, hardware analysis, or experiment.